The Nobel Prize has been among the most prestigious international awards since 1901. Each year, awards are bestowed in chemistry, literature, physics, physiology or medicine, economics, and peace. In addition to the honor, prestige, and substantial prize money, the recipient also gets a gold medal with an image of Alfred Nobel (1833 - 1896), who established the prize.

![](Nobel_Prize.png)

The Nobel Foundation has made a dataset available of all prize winners from the outset of the awards from 1901 to 2023. The dataset used in this project is from the Nobel Prize API and is available in the `nobel.csv` file in the `data` folder.

In this project, you'll get a chance to explore and answer several questions related to this prizewinning data. And we encourage you then to explore further questions that you're interested in!

In [1]:
# ── Importing required libraries ──────────────────────────────────────────────
import pandas as pd          # Data manipulation and analysis
import seaborn as sns        # Statistical data visualization
import numpy as np           # Numerical computing
import matplotlib.pyplot as plt  # Plotting and chart rendering

In [142]:
# ── Load the dataset ───────────────────────────────────────────────────────────
df = pd.read_csv('data/nobel.csv')  # Read Nobel Prize data from CSV into a DataFrame

# plt.show() kept here as a placeholder for any previously rendered plots
plt.show()

# Find the most common gender among all Nobel Prize winners
top_gender = df['sex'].value_counts().idxmax()  # Returns 'Male' or 'Female'

# Find the most common birth country among all Nobel Prize winners
top_country = df['birth_country'].value_counts().idxmax()

# Preview all column names to understand the dataset structure
df.columns

Index(['year', 'category', 'prize', 'motivation', 'prize_share', 'laureate_id',
       'laureate_type', 'full_name', 'birth_date', 'birth_city',
       'birth_country', 'sex', 'organization_name', 'organization_city',
       'organization_country', 'death_date', 'death_city', 'death_country'],
      dtype='object')

In [143]:
# ── Identify the decade with the highest proportion of US-born winners ─────────

# Create a binary flag: 1 if the laureate was born in the USA, 0 otherwise
df['Flag'] = np.where(df['birth_country'] == 'United States of America', 1, 0)

# Derive the decade from the award year (e.g. 1987 → 1980)
df['decade'] = (df['year'] // 10) * 10

# Group by decade and calculate the mean of the flag (= proportion of US winners per decade)
# idxmax() returns the decade index with the highest proportion
max_decade_usa = df.groupby('decade')['Flag'].mean().idxmax()

In [144]:
# ── Find the decade-category combo with the highest share of female winners ────

# Create a binary flag: 1 if the laureate is Female, 0 otherwise
df['female_winner'] = np.where(df['sex'] == 'Female', 1, 0)

# Group by decade AND category, then compute mean female_winner
# (proportion of female laureates within each decade-category pair)
female_winner = df.groupby(['decade', 'category'], as_index=False)['female_winner'].mean()

# Sort descending so the highest female proportion appears first
female_winner = female_winner.sort_values(by='female_winner', ascending=False)

# Build a dict mapping the top decade → its top category (e.g. {2020: 'Literature'})
max_female_dict = {
    female_winner['decade'].values[0]: female_winner['category'].values[0]
}
max_female_dict

{2020: 'Literature'}

In [145]:
# ── Identify the first woman to win a Nobel Prize ─────────────────────────────

# Filter the DataFrame to female laureates only (subset created earlier as `women`)
first_woman = women[['full_name', 'category', 'year']]

# Sort chronologically and reset index, then grab the very first row
first_woman = first_woman.sort_values('year').reset_index(drop=True).iloc[0, :]

# Extract name and category from the resulting Series
first_woman_name = first_woman.values[0]      # e.g. 'Marie Curie, née Sklodowska'
first_woman_category = first_woman.values[1]  # e.g. 'Physics'

print(first_woman_category, first_woman_name)

Physics Marie Curie, née Sklodowska


In [146]:
# ── Build a list of repeat Nobel Prize winners ────────────────────────────────

# Count how many prizes each individual laureate received
count = df.groupby('full_name', as_index=False)['full_name'].value_counts()

# Sort by count descending to see the most decorated laureates at the top
count2 = count.sort_values(by='count', ascending=False).reset_index(drop=True)

# Keep only those who won more than once (count >= 2)
count3 = count[count['count'] >= 2]

# Extract their names into a plain Python list
repeat_list = count3['full_name'].tolist()
repeat_list

['Comité international de la Croix Rouge (International Committee of the Red Cross)',
 'Frederick Sanger',
 'John Bardeen',
 'Linus Carl Pauling',
 'Marie Curie, née Sklodowska',
 'Office of the United Nations High Commissioner for Refugees (UNHCR)']